# Week 2 — Checkpoint 2: Formulation of Null/Alternative Hypotheses
### Dataset: `cars.csv` (turbo.az — car sale listings, cleaned version)

https://www.kaggle.com/datasets/sehriyarmemmedli/turboaz-cars-project

**Goal:** Formulate formal null (H₀) and alternative (H₁) hypotheses for 2 concrete business questions, explain why each matters for the business, and justify which statistical test will be selected.

**Important principle:** When formulating hypotheses, it's essential not to confuse **correlation with causation**. For example, "automatic-transmission cars are more expensive" is a different claim from "adding an automatic transmission increases the price" — our hypotheses will always be about **association/difference**, not **causation** (reason: this dataset is observational, not an experiment — many hidden (confounding) factors could be at play, e.g. automatic transmission tends to appear in newer/luxury models).

## 1. Restoring the cleaned dataset (from Checkpoint 1)

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew

pd.set_option('display.max_columns', 40)

cols = [
    "id_x", "car_rel_url_x", "datetime_scrape", "price_x", "currency_x", "city",
    "production_year", "engine_displacement_num", "kilometrage_num", "Marka", "Model",
    "Sürətlər qutusu", "Vəziyyəti", "Ötürücü", "Ban növü", "views"
]

df = pd.read_csv("cars.csv", usecols=cols, parse_dates=["datetime_scrape"])
df_dedup = df.sort_values("datetime_scrape").drop_duplicates(subset="car_rel_url_x", keep="last").copy()

exchange_rate = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df_dedup["price_azn"] = df_dedup["price_x"] * df_dedup["currency_x"].map(exchange_rate)

exclude_body_types = ["Yük maşını", "Motosiklet", "Avtobus", "Moped", "Kvadrosikl", "Dartqı", "Mikroavtobus"]
df_clean = df_dedup[~df_dedup["Ban növü"].isin(exclude_body_types)].copy()
df_clean = df_clean[df_clean["price_azn"] >= 1000].copy()

print("Clean dataset shape:", df_clean.shape)


Clean dataset shape: (149478, 17)


## 2. Business Question 1 — Does gearbox type (manual / automatic) affect price?

### Business context
A practical question useful for both sellers and buyers on turbo.az: **"Do automatic-transmission cars sell for a significantly different average price than manual-transmission cars?"** This matters both for a seller pricing a listing and a buyer planning a budget.

### Variables
- **Dependent variable (continuous):** `price_azn` — the car's price (AZN)
- **Independent variable (categorical, 2 levels):** `Sürətlər qutusu` (gearbox) — `Mexaniki` (manual) vs `Avtomat` (automatic)

### Hypotheses
- **H₀ (null hypothesis):** There is **no difference in the population mean price** between manual and automatic transmission cars.
  $$H_0: \mu_{automatic} = \mu_{manual}$$
- **H₁ (alternative hypothesis):** There **is a difference in the population mean price** between manual and automatic transmission cars (two-tailed test — we don't assume the direction in advance, we test it statistically).
  $$H_1: \mu_{automatic} \neq \mu_{manual}$$

### Significance level
$$\alpha = 0.05$$

### Test selection (justification)
Since the dependent variable is **continuous** and the independent variable is **categorical with 2 levels**, the appropriate test is an **independent samples t-test**. Normality and equality of variance will be checked in Checkpoint 5 — if those assumptions are violated, Welch's t-test or the Mann-Whitney U test (non-parametric alternative) can be used instead.

In [2]:
gearbox_preview = df_clean[df_clean["Sürətlər qutusu"].isin(["Mexaniki", "Avtomat"])].groupby("Sürətlər qutusu")["price_azn"].agg(["count", "mean", "median"])
gearbox_preview.round(0)


,count,mean,median
Sürətlər qutusu,,,
Avtomat,98161,30023.0,22000.0
Mexaniki,37161,10567.0,8900.0


**Correlation vs. causation warning:** If we find a statistically significant difference in this test, it will **not** mean "choosing an automatic transmission increases the price." The real cause is more likely a **confounding variable** — automatic transmission is more common in newer, luxury, higher-engine-displacement models, and those models are inherently more expensive on their own. The relationship between gearbox type and price is likely **indirect** (gearbox → car segment → price), not a direct causal link.

## 3. Business Question 2 — Is brand a factor in car price, and which brands differ from each other?

### Business context
Among the 5 most commonly sold brands (Mercedes, Hyundai, Kia, Toyota, LADA), is there a **significant price difference**? This is useful both for the platform's price-recommendation systems and for answering a consumer's "which brand is the better deal" question.

**Note:** This question requires comparing **5 groups** — unlike a simple pair (2-group) comparison, this creates a **multiple comparisons problem**: if we ran a separate t-test for every pair of brands (5 brands = 10 pairwise comparisons), the probability of finding at least one "significant" result by pure chance — even if no real difference exists — rises sharply (false-positive risk). This problem will be addressed in Checkpoint 3 (with Bonferroni correction).

### Variables
- **Dependent variable (continuous):** `price_azn`
- **Independent variable (categorical, 5 levels):** `Marka` (brand) — Mercedes, Hyundai, Kia, Toyota, LADA (VAZ)

### Hypotheses
- **H₀ (null hypothesis):** All 5 brands have **equal population mean price** (no difference between any brands).
  $$H_0: \mu_{Mercedes} = \mu_{Hyundai} = \mu_{Kia} = \mu_{Toyota} = \mu_{LADA}$$
- **H₁ (alternative hypothesis):** **At least one** brand's population mean price differs from the others (we reject the claim that all brands are equal, but ANOVA itself doesn't tell us WHICH pairs differ — that requires a post-hoc test).
  $$H_1: \text{at least one } \mu_i \neq \mu_j$$

### Significance level
$$\alpha = 0.05 \text{ (for ANOVA); Bonferroni-corrected } \alpha \text{ for post-hoc pairwise comparisons}$$

### Test selection (justification)
Since the dependent variable is **continuous** and the independent variable is **categorical with more than 2 levels (5)**, the appropriate test is a **one-way ANOVA** (rather than a series of separate t-tests — since multiple t-tests increase the false-positive risk). If ANOVA rejects H₀, **Bonferroni-corrected post-hoc pairwise t-tests** will be run to see **which** pairs differ.

In [3]:
top5_brands = ["Mercedes", "Hyundai", "Kia", "Toyota", "LADA (VAZ)"]
brand_preview = df_clean[df_clean["Marka"].isin(top5_brands)].groupby("Marka")["price_azn"].agg(["count", "mean", "median"]).loc[top5_brands]
brand_preview.round(0)


,count,mean,median
Marka,,,
Mercedes,25307,25650.0,14300.0
Hyundai,19658,23864.0,22700.0
Kia,14619,25765.0,24500.0
Toyota,14388,30287.0,23200.0
LADA (VAZ),12859,7264.0,6100.0


In [4]:
from itertools import combinations
n_groups = len(top5_brands)
n_pairs = len(list(combinations(top5_brands, 2)))
print(f"Number of possible pairwise comparisons for {n_groups} groups: {n_pairs}")
print(f"If we test each one separately at α=0.05, probability of at least one false positive: {1-(1-0.05)**n_pairs:.1%}")
print(f"(Bonferroni-corrected α per test: {0.05/n_pairs:.4f})")


Number of possible pairwise comparisons for 5 groups: 10
If we test each one separately at α=0.05, probability of at least one false positive: 40.1%
(Bonferroni-corrected α per test: 0.0050)


**Explanation of the number:** If we test all 10 pairwise comparisons separately at α=0.05 each, **even if no real difference exists**, the probability of finding at least one "significant" (false) result reaches ~40% — unacceptably high. Bonferroni correction controls this risk by lowering each test's α to 0.05/10 = 0.005. This will be applied in Checkpoint 3.

## Checkpoint 2 — Summary

| | Business Question 1 | Business Question 2 |
|---|---|---|
| **Question** | Is gearbox type associated with price? | Is brand associated with price? |
| **Groups** | 2 (Manual / Automatic) | 5 (Mercedes, Hyundai, Kia, Toyota, LADA) |
| **H₀** | μ_automatic = μ_manual | μ₁ = μ₂ = μ₃ = μ₄ = μ₅ |
| **H₁** | μ_automatic ≠ μ_manual | at least one pair differs |
| **Test** | Independent samples t-test | One-way ANOVA + Bonferroni post-hoc |
| **α** | 0.05 | 0.05 (ANOVA), 0.005 (post-hoc, corrected) |
| **Specific risk** | — | Multiple comparisons problem |

Results for both questions will be interpreted as **association/difference**, not **causation** — the dataset is observational and confounding variables are present.
